# 05 · Resolve links, assign master keys, survivorship, DQ gate, publish Gold

Reads steward decisions from `rvw_decision_log` (shortcut to the mirrored SQL database), builds clusters, applies the DQ gate and publishes Gold as SCD2. If a blocking rule fails, Gold is not touched and the notebook fails so the pipeline alerts.

Attach the **pipeline Lakehouse** as default and the **goldenrecord Environment** (wheel + config). All logic is in the `goldenrecord` library; this notebook only calls it.

In [ ]:
# Parameters (overridden by the pipeline / orchestrator)
run_id = ""                                      # same id for every stage of one run
config_dir = "/lakehouse/default/Files/config"   # config/ folder uploaded to the Lakehouse
landing_root = "Files/landing"                   # where source extracts land
use_key_vault = True                             # False only in a sandbox without Key Vault
use_fabric_ai = True                             # False -> deterministic heuristic (no AI calls)

In [ ]:
from goldenrecord.fabric_entry import exit_with, fabric_context

ctx = fabric_context(
    spark,
    config_dir=config_dir,
    landing_root=landing_root,
    use_key_vault=use_key_vault,
    use_fabric_ai=use_fabric_ai,
    run_id=run_id or None,
)
print("run_id:", ctx.run_id)

In [ ]:
from goldenrecord.jobs import run_resolve

result = run_resolve(ctx)
if not result['gate_passed']:
    raise RuntimeError(f"DQ gate failed, Gold not published: {result['failed_rules']}")
result

In [ ]:
exit_with(result)